In [1]:
import pandas as pd
import matplotlib.pyplot as plt 

df = pd.read_csv('gyro_acc_resampled_sliding_window.csv')

df.head()

,t,accelerometerX,accelerometerY,accelerometerZ,gyroscopeX,gyroscopeY,gyroscopeZ,label,dt,dt_seconds,session_break,session_id
0,0 days 00:00:57.838720972,-0.835000,5.255000,8.400000,0.000000,-0.410000,0.140000,STANDING,0 days 00:00:00,0.00,False,0
1,0 days 00:00:57.858720972,-0.870000,5.400000,8.670000,0.000000,-0.410000,0.140000,STANDING,0 days 00:00:00.020000,0.02,False,0
2,0 days 00:00:57.878720972,-1.237143,5.412857,8.914286,-0.152857,-0.194286,0.032857,STANDING,0 days 00:00:00.020000,0.02,False,0
3,0 days 00:00:57.898720972,-1.604286,5.425714,9.158571,-0.305714,0.021429,-0.074286,STANDING,0 days 00:00:00.020000,0.02,False,0
4,0 days 00:00:57.918720972,-1.971428,5.438571,9.402857,-0.458571,0.237143,-0.181429,STANDING,0 days 00:00:00.020000,0.02,False,0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39843 entries, 0 to 39842
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   t               39843 non-null  object 
 1   accelerometerX  39843 non-null  float64
 2   accelerometerY  39843 non-null  float64
 3   accelerometerZ  39843 non-null  float64
 4   gyroscopeX      39843 non-null  float64
 5   gyroscopeY      39843 non-null  float64
 6   gyroscopeZ      39843 non-null  float64
 7   label           39843 non-null  object 
 8   dt              39843 non-null  object 
 9   dt_seconds      39843 non-null  float64
 10  session_break   39843 non-null  bool   
 11  session_id      39843 non-null  int64  
dtypes: bool(1), float64(7), int64(1), object(3)
memory usage: 3.4+ MB


In [3]:
df.shape

(39843, 12)

In [4]:
# convert t column back into Timedelta if needed
df["t"] = pd.to_timedelta(df["t"])

# set time as index
df = df.set_index("t").sort_index()

In [6]:
import numpy as np
import pandas as pd

sensor_cols = [
    "accelerometerX","accelerometerY","accelerometerZ",
    "gyroscopeX","gyroscopeY","gyroscopeZ",
]

WINDOW_SIZE = 128
STEP_SIZE   = 64

X_windows = []
y_windows = []

for sid, group in df.groupby("session_id"):

    # drop rows with missing sensor values
    group = group.dropna(subset=sensor_cols)

    if len(group) < WINDOW_SIZE:
        continue

    values = group[sensor_cols].to_numpy()
    labels = group["label"].to_numpy()

    for start in range(0, len(group) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        
        window_vals = values[start:end]
        window_labels = labels[start:end]

        # handle missing labels safely
        label_series = pd.Series(window_labels).dropna()
        if label_series.empty:
            continue

        # majority label in window
        window_label = label_series.mode().iloc[0]

        X_windows.append(window_vals)
        y_windows.append(window_label)

X = np.stack(X_windows)
y = np.array(y_windows)

print("Windows:", X.shape)
print("Unique labels:", np.unique(y))


Windows: (617, 128, 6)
Unique labels: ['SITTING' 'STANDING' 'WALKING' 'WALKING_DOWNSTAIRS']


In [29]:
# helper functions

import numpy as np

def correlation(x, y):
    return np.corrcoef(x, y)[0,1] if np.std(x) > 0 and np.std(y) > 0 else 0


import numpy as np
from scipy.fft import rfft, rfftfreq

DT = 0.02  # 50 Hz
GRAVITY = 9.80665  # standard gravity in m/s²

def energy(sig):
    """Calculate normalized energy of signal"""
    sig = np.asarray(sig)
    if len(sig) == 0:
        return 0.0
    return np.sum(sig**2) / len(sig)

def entropy(sig, bins=10):
    """Calculate entropy with safeguards against extreme values"""
    sig = np.asarray(sig)
    if len(sig) == 0 or np.std(sig) == 0:
        return 0.0
    
    hist, _ = np.histogram(sig, bins=bins, density=False)
    # Normalize to get probabilities
    hist = hist / hist.sum()
    # Filter out zeros and add small epsilon to prevent log(0)
    hist = hist[hist > 0]
    
    if len(hist) == 0:
        return 0.0
    
    # Clip entropy to reasonable range to prevent extreme values
    ent = -np.sum(hist * np.log2(hist + 1e-10))
    return np.clip(ent, 0, 10)  # entropy shouldn't exceed ~10 for 10 bins

def sma(x, y, z):
    x, y, z = map(np.asarray, (x, y, z))
    return np.sum(np.abs(x) + np.abs(y) + np.abs(z)) / len(x)

def mean_freq(spec, freqs):
    spec = np.asarray(spec)
    freqs = np.asarray(freqs)
    if spec.sum() == 0:
        return 0.0
    return np.sum(freqs * spec) / np.sum(spec)

## Fixed Issues to Match Kaggle Dataset

Three critical fixes applied:

1. **Entropy calculation**: Added safeguards against log(0) and clipped extreme values to [0, 10] range
2. **Accelerometer normalization**: Divided by standard gravity (9.80665 m/s²) to get values in 'g' units matching Kaggle's [-1, 1] range
3. **Energy normalization**: Now properly normalized by signal length

These changes ensure feature values match the Kaggle HAR dataset's preprocessing.

In [11]:
# build time-domain features

import pandas as pd

def extract_time_features(window):
    accx, accy, accz, gyx, gyy, gyz = window.T
    
    features = {}

    signals = {
        "tBodyAcc_X": accx,
        "tBodyAcc_Y": accy,
        "tBodyAcc_Z": accz,
        "tBodyGyro_X": gyx,
        "tBodyGyro_Y": gyy,
        "tBodyGyro_Z": gyz
    }

    for name, sig in signals.items():
        feature_name = name.split('_')[0]
        axis = name.split('_')[1]
        features[f"{feature_name}-mean()-{axis}"] = sig.mean()
        features[f"{feature_name}-std()-{axis}"]  = sig.std()
        features[f"{feature_name}-min()-{axis}"]  = sig.min()
        features[f"{feature_name}-max()-{axis}"]  = sig.max()
        features[f"{feature_name}-energy()-{axis}"] = energy(sig)
        features[f"{feature_name}-entropy()-{axis}"] = entropy(sig)

    # magnitude (like AccMag in HAR)
    acc_mag = np.sqrt(accx**2 + accy**2 + accz**2)
    gyro_mag = np.sqrt(gyx**2 + gyy**2 + gyz**2)

    features["tBodyAcc-sma()"]  = sma(accx, accy, accz)
    features["tBodyGyro-sma()"] = sma(gyx, gyy, gyz)

    # correlations
    features["tBodyAcc-correlation()-X,Y"] = correlation(accx, accy)
    features["tBodyAcc-correlation()-X,Z"] = correlation(accx, accz)
    features["tBodyAcc-correlation()-Y,Z"] = correlation(accy, accz)

    features["tBodyGyro-correlation()-X,Y"] = correlation(gyx, gyy)
    features["tBodyGyro-correlation()-X,Z"] = correlation(gyx, gyz)
    features["tBodyGyro-correlation()-Y,Z"] = correlation(gyy, gyz)
    return features

In [30]:
## Step 3 — FFT (frequency-domain features)
# HAR applies FFT to each signal:

from scipy.fft import rfft
DT = 0.02   # 50 Hz
FS = 50.0
def extract_freq_features(window):
    accx, accy, accz, gyx, gyy, gyz = window.T

    features = {}
    signals = {
        "fBodyAcc_X": np.abs(rfft(accx)),
        "fBodyAcc_Y": np.abs(rfft(accy)),
        "fBodyAcc_Z": np.abs(rfft(accz)),
        "fBodyGyro_X": np.abs(rfft(gyx)),
        "fBodyGyro_Y": np.abs(rfft(gyy)),
        "fBodyGyro_Z": np.abs(rfft(gyz)),
    }

    for name, spec in signals.items():
        feature_name = name.split('_')[0]
        axis = name.split('_')[1]
        features[f"{feature_name}-mean()-{axis}"] = spec.mean()
        features[f"{feature_name}-std()-{axis}"]  = spec.std()
        features[f"{feature_name}-energy()-{axis}"] = energy(spec)
        features[f"{feature_name}-entropy()-{axis}"] = entropy(spec)

    return features

from scipy.signal import butter, filtfilt

def lowpass_filter(sig, cutoff=0.3, fs=FS, order=4):
    """Butterworth low-pass like UCI HAR (~0.3 Hz to isolate gravity)."""
    sig = np.asarray(sig)
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return filtfilt(b, a, sig)

def angle_between(v1, v2):
    """Angle between 2 vectors in radians."""
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    cosang = np.dot(v1, v2) / (n1 * n2)
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.arccos(cosang)

In [31]:
def extract_features(window, dt=DT, fs=FS):
    """
    window: np.array (128, 6) [accX, accY, accZ, gyroX, gyroY, gyroZ]
    returns: dict of HAR-style features (Body + Gravity + Angles)
    """
    # *** NORMALIZE TO MATCH KAGGLE DATASET ***
    # Divide accelerometer by gravity to get values in 'g' units (like Kaggle: ~-1 to 1)
    accx = window[:, 0] / GRAVITY
    accy = window[:, 1] / GRAVITY
    accz = window[:, 2] / GRAVITY
    gyx = window[:, 3]
    gyy = window[:, 4]
    gyz = window[:, 5]
    
    features = {}

    # ---------- GRAVITY vs BODY SPLIT ----------
    gravity_x = lowpass_filter(accx, cutoff=0.3, fs=fs)
    gravity_y = lowpass_filter(accy, cutoff=0.3, fs=fs)
    gravity_z = lowpass_filter(accz, cutoff=0.3, fs=fs)

    body_x = accx - gravity_x
    body_y = accy - gravity_y
    body_z = accz - gravity_z

    # ---------- TIME-DOMAIN: BODY ACC & GYRO ----------
    time_signals = {
        "tBodyAcc-X": body_x,
        "tBodyAcc-Y": body_y,
        "tBodyAcc-Z": body_z,
        "tBodyGyro-X": gyx,
        "tBodyGyro-Y": gyy,
        "tBodyGyro-Z": gyz,
    }

    for name, sig in time_signals.items():
        sig = np.asarray(sig)
        base, axis = name.split('-')   # "tBodyAcc", "X"

        features[f"{base}-mean()-{axis}"]   = sig.mean()
        features[f"{base}-std()-{axis}"]    = sig.std()
        features[f"{base}-min()-{axis}"]    = sig.min()
        features[f"{base}-max()-{axis}"]    = sig.max()
        features[f"{base}-energy()-{axis}"] = energy(sig)
        features[f"{base}-entropy()-{axis}"] = entropy(sig)

    # ---------- TIME-DOMAIN: GRAVITY ACC ----------
    grav_signals = {
        "tGravityAcc-X": gravity_x,
        "tGravityAcc-Y": gravity_y,
        "tGravityAcc-Z": gravity_z,
    }

    for name, sig in grav_signals.items():
        sig = np.asarray(sig)
        base, axis = name.split('-')
        features[f"{base}-mean()-{axis}"]   = sig.mean()
        features[f"{base}-std()-{axis}"]    = sig.std()
        features[f"{base}-min()-{axis}"]    = sig.min()
        features[f"{base}-max()-{axis}"]    = sig.max()
        features[f"{base}-energy()-{axis}"] = energy(sig)
        features[f"{base}-entropy()-{axis}"] = entropy(sig)

    # ---------- TIME-DOMAIN: JERK (body acc + gyro) ----------
    body_x_j = np.diff(body_x) / dt
    body_y_j = np.diff(body_y) / dt
    body_z_j = np.diff(body_z) / dt

    gyx_j = np.diff(gyx) / dt
    gyy_j = np.diff(gyy) / dt
    gyz_j = np.diff(gyz) / dt

    jerk_time_signals = {
        "tBodyAccJerk-X": body_x_j,
        "tBodyAccJerk-Y": body_y_j,
        "tBodyAccJerk-Z": body_z_j,
        "tBodyGyroJerk-X": gyx_j,
        "tBodyGyroJerk-Y": gyy_j,
        "tBodyGyroJerk-Z": gyz_j,
    }

    for name, sig in jerk_time_signals.items():
        sig = np.asarray(sig)
        base, axis = name.split('-')
        features[f"{base}-mean()-{axis}"]   = sig.mean()
        features[f"{base}-std()-{axis}"]    = sig.std()
        features[f"{base}-min()-{axis}"]    = sig.min()
        features[f"{base}-max()-{axis}"]    = sig.max()
        features[f"{base}-energy()-{axis}"] = energy(sig)
        features[f"{base}-entropy()-{axis}"] = entropy(sig)

    # ---------- TIME-DOMAIN: MAGNITUDES ----------
    body_mag       = np.sqrt(body_x**2      + body_y**2      + body_z**2)
    grav_mag       = np.sqrt(gravity_x**2   + gravity_y**2   + gravity_z**2)
    body_jerk_mag  = np.sqrt(body_x_j**2    + body_y_j**2    + body_z_j**2)
    gyro_mag       = np.sqrt(gyx**2         + gyy**2         + gyz**2)
    gyro_jerk_mag  = np.sqrt(gyx_j**2       + gyy_j**2       + gyz_j**2)

    mag_signals = {
        "tBodyAccMag":      body_mag,
        "tGravityAccMag":   grav_mag,
        "tBodyAccJerkMag":  body_jerk_mag,
        "tBodyGyroMag":     gyro_mag,
        "tBodyGyroJerkMag": gyro_jerk_mag,
    }

    for base, sig in mag_signals.items():
        sig = np.asarray(sig)
        features[f"{base}-mean()"]   = sig.mean()
        features[f"{base}-std()"]    = sig.std()
        features[f"{base}-min()"]    = sig.min()
        features[f"{base}-max()"]    = sig.max()
        features[f"{base}-energy()"] = energy(sig)
        features[f"{base}-entropy()"] = entropy(sig)
        features[f"{base}-sma()"]    = np.sum(np.abs(sig)) / len(sig)

    # ---------- ANGLES (like UCI HAR) ----------
    body_acc_mean   = np.array([body_x.mean(),      body_y.mean(),      body_z.mean()])
    gravity_mean    = np.array([gravity_x.mean(),   gravity_y.mean(),   gravity_z.mean()])
    body_acc_j_mean = np.array([body_x_j.mean(),    body_y_j.mean(),    body_z_j.mean()])
    gyro_mean       = np.array([gyx.mean(),         gyy.mean(),         gyz.mean()])
    gyro_j_mean     = np.array([gyx_j.mean(),       gyy_j.mean(),       gyz_j.mean()])

    # angles between mean vectors and gravity
    features["angle(tBodyAccMean,gravityMean)"]      = angle_between(body_acc_mean, gravity_mean)
    features["angle(tBodyAccJerkMean,gravityMean)"]  = angle_between(body_acc_j_mean, gravity_mean)
    features["angle(tBodyGyroMean,gravityMean)"]     = angle_between(gyro_mean, gravity_mean)
    features["angle(tBodyGyroJerkMean,gravityMean)"] = angle_between(gyro_j_mean, gravity_mean)

    # angles between gravity and unit axes (approx HAR's angle(X,gravityMean), etc.)
    x_unit = np.array([1.0, 0.0, 0.0])
    y_unit = np.array([0.0, 1.0, 0.0])
    z_unit = np.array([0.0, 0.0, 1.0])

    features["angle(X,gravityMean)"] = angle_between(x_unit, gravity_mean)
    features["angle(Y,gravityMean)"] = angle_between(y_unit, gravity_mean)
    features["angle(Z,gravityMean)"] = angle_between(z_unit, gravity_mean)

    # ---------- FREQUENCY-DOMAIN (Body / Jerk / Mags as before) ----------
    N_acc  = len(body_x)
    N_j    = len(body_x_j)
    freqs_acc = rfftfreq(N_acc, d=dt)
    freqs_j   = rfftfreq(N_j, d=dt)

    freq_signals = {
        "fBodyAcc-X":      (np.abs(rfft(body_x)),      freqs_acc),
        "fBodyAcc-Y":      (np.abs(rfft(body_y)),      freqs_acc),
        "fBodyAcc-Z":      (np.abs(rfft(body_z)),      freqs_acc),

        "fBodyGyro-X":     (np.abs(rfft(gyx)),         freqs_acc),
        "fBodyGyro-Y":     (np.abs(rfft(gyy)),         freqs_acc),
        "fBodyGyro-Z":     (np.abs(rfft(gyz)),         freqs_acc),

        "fBodyAccJerk-X":  (np.abs(rfft(body_x_j)),    freqs_j),
        "fBodyAccJerk-Y":  (np.abs(rfft(body_y_j)),    freqs_j),
        "fBodyAccJerk-Z":  (np.abs(rfft(body_z_j)),    freqs_j),

        "fBodyGyroJerk-X": (np.abs(rfft(gyx_j)),       freqs_j),
        "fBodyGyroJerk-Y": (np.abs(rfft(gyy_j)),       freqs_j),
        "fBodyGyroJerk-Z": (np.abs(rfft(gyz_j)),       freqs_j),
    }

    for name, (spec, f) in freq_signals.items():
        base, axis = name.split('-')
        features[f"{base}-mean()-{axis}"]      = spec.mean()
        features[f"{base}-std()-{axis}"]       = spec.std()
        features[f"{base}-energy()-{axis}"]    = energy(spec)
        features[f"{base}-entropy()-{axis}"]   = entropy(spec)
        features[f"{base}-meanFreq()-{axis}"]  = mean_freq(spec, f)

    freq_mag_signals = {
        "fBodyAccMag":      (np.abs(rfft(body_mag)),       freqs_acc),
        "fBodyAccJerkMag":  (np.abs(rfft(body_jerk_mag)),  freqs_j),
        "fBodyGyroMag":     (np.abs(rfft(gyro_mag)),       freqs_acc),
        "fBodyGyroJerkMag": (np.abs(rfft(gyro_jerk_mag)),  freqs_j),
    }

    for base, (spec, f) in freq_mag_signals.items():
        features[f"{base}-mean()"]      = spec.mean()
        features[f"{base}-std()"]       = spec.std()
        features[f"{base}-energy()"]    = energy(spec)
        features[f"{base}-entropy()"]   = entropy(spec)
        features[f"{base}-meanFreq()"]  = mean_freq(spec, f)

    return features


In [32]:
##Step 4 — apply BOTH to all windows
feature_rows = []

for w in X:
    feats = {}
    feats.update(extract_time_features(w))
    feats.update(extract_freq_features(w))
    feature_rows.append(feats)

X_features = pd.DataFrame(feature_rows)
y_features = pd.Series(y)

print(X_features.shape)
X_features.head()

feature_rows = [extract_features(w) for w in X]
X_features = pd.DataFrame(feature_rows)
y_features = pd.Series(y)

print(X_features.shape)
print(X_features.columns[:])

(617, 68)
(617, 212)
Index(['tBodyAcc-mean()-X', 'tBodyAcc-std()-X', 'tBodyAcc-min()-X',
       'tBodyAcc-max()-X', 'tBodyAcc-energy()-X', 'tBodyAcc-entropy()-X',
       'tBodyAcc-mean()-Y', 'tBodyAcc-std()-Y', 'tBodyAcc-min()-Y',
       'tBodyAcc-max()-Y',
       ...
       'fBodyGyroMag-mean()', 'fBodyGyroMag-std()', 'fBodyGyroMag-energy()',
       'fBodyGyroMag-entropy()', 'fBodyGyroMag-meanFreq()',
       'fBodyGyroJerkMag-mean()', 'fBodyGyroJerkMag-std()',
       'fBodyGyroJerkMag-energy()', 'fBodyGyroJerkMag-entropy()',
       'fBodyGyroJerkMag-meanFreq()'],
      dtype='object', length=212)


In [26]:
X_features.head()

,tBodyAcc-mean()-X,tBodyAcc-std()-X,tBodyAcc-min()-X,tBodyAcc-max()-X,tBodyAcc-energy()-X,tBodyAcc-entropy()-X,tBodyAcc-mean()-Y,tBodyAcc-std()-Y,tBodyAcc-min()-Y,tBodyAcc-max()-Y,...,fBodyGyroMag-mean(),fBodyGyroMag-std(),fBodyGyroMag-energy(),fBodyGyroMag-entropy(),fBodyGyroMag-meanFreq(),fBodyGyroJerkMag-mean(),fBodyGyroJerkMag-std(),fBodyGyroJerkMag-energy(),fBodyGyroJerkMag-entropy(),fBodyGyroJerkMag-meanFreq()
0,-0.480865,1.124545,-2.743870,3.981376,1.495834,3.144261,0.305431,1.343796,-3.822451,4.732531,...,8.172581,20.742604,497.046709,0.281107,4.879113,320.145906,364.644965,235459.351647,0.032122,9.610271
1,0.064124,1.547782,-3.446329,2.663288,2.399740,3.862915,0.219118,1.290055,-2.687009,3.996720,...,4.603638,10.183367,124.894437,0.482900,5.280982,151.055706,168.856567,51330.366650,0.063613,9.601974
2,0.932877,1.931551,-2.488029,6.621432,4.601151,2.776653,-0.078374,1.471979,-2.880757,4.940870,...,8.478049,17.931015,393.398632,0.356587,4.277960,180.732863,185.868869,67211.604474,0.061934,9.739804
3,0.842127,1.553607,-2.237545,6.667856,3.122872,2.829998,0.075546,1.178647,-3.233219,3.677620,...,9.027260,26.449433,781.063946,0.233828,4.270597,249.688508,307.353871,156810.752914,0.036971,10.148589
4,0.028172,1.501514,-3.966639,5.159842,2.255339,2.740736,0.036184,1.240071,-3.018120,3.342904,...,7.942334,24.048815,641.426157,0.243638,4.497831,247.799944,338.385898,175909.828322,0.032535,10.438297


In [27]:
y_features

0      STANDING
1      STANDING
2      STANDING
3      STANDING
4      STANDING
         ...   
612    STANDING
613    STANDING
614    STANDING
615    STANDING
616    STANDING
Length: 617, dtype: object

In [33]:
data_to_export = X_features.copy()
data_to_export["label"] = y_features

data_to_export.to_csv("gyro_acc_har_features.csv", index=False)

In [34]:
# Verify the fixes - check value ranges
print("=== VERIFICATION OF FIXES ===\n")

# Check accelerometer features (should be ~-1 to 1 now)
acc_features = X_features.filter(like='tBodyAcc-mean()')
print("Body Acceleration means (should be ~-1 to 1):")
print(acc_features.describe())

# Check entropy features (should be 0-10, no extreme negatives)
entropy_features = X_features.filter(like='entropy')
print("\nEntropy features (should be 0-10, no extreme values):")
print(f"Min: {entropy_features.min().min():.2f}")
print(f"Max: {entropy_features.max().max():.2f}")
print(f"Any extreme values (>100 or <-100): {((entropy_features.abs() > 100).sum().sum())}")

# Check overall value ranges
print(f"\n=== OVERALL VALUE RANGES ===")
print(f"Dataset min: {X_features.min().min():.6f}")
print(f"Dataset max: {X_features.max().max():.6f}")

# Check for problematic features
print(f"\nFeatures with values > 100 or < -100:")
extreme_count = 0
for col in X_features.columns:
    col_min, col_max = X_features[col].min(), X_features[col].max()
    if abs(col_min) > 100 or abs(col_max) > 100:
        extreme_count += 1
        print(f"  {col}: min={col_min:.2e}, max={col_max:.2e}")

if extreme_count == 0:
    print("  ✓ None! All features are in reasonable ranges.")

=== VERIFICATION OF FIXES ===

Body Acceleration means (should be ~-1 to 1):
       tBodyAcc-mean()-X  tBodyAcc-mean()-Y  tBodyAcc-mean()-Z
count       6.170000e+02         617.000000         617.000000
mean        5.453458e-05           0.000416          -0.000483
std         1.944697e-02           0.017570           0.022368
min        -1.204165e-01          -0.240845          -0.162980
25%        -3.948696e-03          -0.001463          -0.004258
50%         1.555746e-07           0.000060           0.000073
75%         4.815555e-03           0.003286           0.004380
max         1.365523e-01           0.077031           0.135237

Entropy features (should be 0-10, no extreme values):
Min: 0.11
Max: 3.32
Any extreme values (>100 or <-100): 0

=== OVERALL VALUE RANGES ===
Dataset min: -261.500000
Dataset max: 663958.309232

Features with values > 100 or < -100:
  tBodyAccJerk-energy()-X: min=1.22e-13, max=3.23e+02
  tBodyAccJerk-energy()-Y: min=8.21e-13, max=2.20e+02
  tBodyAccJerk

## Analysis

**Major improvements:**
- ✓ Entropy values are now in reasonable range (0.11 - 3.32) instead of extreme negatives
- ✓ Body acceleration means are properly normalized (~-0.12 to 0.14)

**Remaining differences from Kaggle:**
- Jerk and frequency domain energy features still have larger values than Kaggle
- This is expected since Kaggle likely applies additional normalization per-subject
- These features should still work with proper scaling in the ML model

The key fixes (entropy, gravity normalization) have been applied. The remaining value differences are manageable with StandardScaler during model training.